# Exploración de resultados — Tesis TFT Electricidad

Notebook curado que consolida los hallazgos principales dispersos en los notebooks de trabajo
(`Multi_modelo TFT *.ipynb`, `Comparativa_*.ipynb`, `Resultados_resumen.ipynb`).

**No reentrena nada.** Cada sección carga resultados ya calculados y guardados —
checkpoints (`eval_TFT_*.pkl`) o cachés consolidados en
[`Multi-Modelos_TFT/Resultados_Consolidados/`](Multi-Modelos_TFT/Resultados_Consolidados/) —
y agrega una interpretación breve.

Este notebook es **aditivo**: los notebooks de trabajo originales no se tocan ni se borran.

## 1. Setup

In [ ]:
import os
import sys
import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RUTA_BASE = r"C:\Users\56977\OneDrive\Escritorio\Tesis - copia"
os.chdir(RUTA_BASE)
sys.path.insert(0, RUTA_BASE)

from Modulos.Comparativa import diebold_mariano_desde_errores, perdida_asimetrica
from Modulos.Evaluacion_TFT import recalcular_metricas_serie_cruda, VENTANA_PRECIO_FALLA

RESULTADOS = "Multi-Modelos_TFT/Resultados_Consolidados"
BARRAS = ["ATACAMA", "CARDONES", "CHARRUA", "CRUCERO", "P.AZUCAR", "P.MONTT", "QUILLOTA", "TARAPACA"]

EXP_SOL_CLIMA = {"LN": "Multi-Modelos_TFT/h/LN/pred_sol_clima", "DyT": "Multi-Modelos_TFT/h/DyT/pred_sol_clima"}
EXP_PROPHET = {"LN": "Multi-Modelos_TFT/h/LN/pred_prophet_clima", "DyT": "Multi-Modelos_TFT/h/DyT/pred_prophet_clima"}


def ruta(carpeta_relativa):
    return os.path.join(RUTA_BASE, carpeta_relativa)


def cargar_json(nombre):
    with open(os.path.join(RESULTADOS, nombre), encoding="utf-8") as f:
        return json.load(f)


def cargar_eval_pkl(carpeta_relativa, arquitectura, barra, conjunto="test"):
    """Carga las predicciones ya calculadas por el modelo entrenado (sin reentrenar)."""
    pkl_path = os.path.join(ruta(carpeta_relativa), "Resultados", f"eval_TFT_{arquitectura}.pkl")
    with open(pkl_path, "rb") as f:
        datos_evaluacion, _ = pickle.load(f)
    d = datos_evaluacion[barra][conjunto]["Precios"]["datos"]
    return pd.DatetimeIndex(d["fechas"]), np.asarray(d["y_real"]), np.asarray(d["y_pred"])


print("Barras:", BARRAS)

## 2. S+C-LN de referencia

El modelo ganador de la tesis: Transformer con LayerNorm, features de posición solar + clima,
sin Prophet (S+C). Métricas recalculadas sobre la **serie cruda** (sin recorte de outliers en
val/test) y excluyendo la ventana de precio de falla administrativo, más MASE/skill score
contra la línea base de persistencia de 1h.

In [ ]:
sc_ln = cargar_json("sc_ln_metricas_baselines.json")

filas = []
for barra in BARRAS:
    d = sc_ln[barra]
    filas.append({
        "Barra": barra,
        "MAE_train": d["train"]["MAE"], "MAE_val": d["val"]["MAE"], "MAE_test": d["test"]["MAE"],
        "R2_test": d["test"]["R2"],
        "MASE": d["baselines"]["MASE"], "Skill_score": d["baselines"]["skill_score"],
    })
df_sc_ln = pd.DataFrame(filas).set_index("Barra").round(3)
df_sc_ln

In [ ]:
BARRA_GRAFICO = "ATACAMA"  # cambiar aca para ver otra barra
ZOOM_INICIO, ZOOM_DIAS = "2025-07-07", 10

fechas, y_real, y_pred = cargar_eval_pkl(EXP_SOL_CLIMA["LN"], "LN", BARRA_GRAFICO, "test")
mask = (fechas >= ZOOM_INICIO) & (fechas < pd.Timestamp(ZOOM_INICIO) + pd.Timedelta(days=ZOOM_DIAS))

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(fechas[mask], y_real[mask], label="Real", color="steelblue", marker="o", markersize=2)
ax.plot(fechas[mask], y_pred[mask], label="Predicción S+C-LN", color="firebrick", marker="o", markersize=2, alpha=0.8)
ax.set_title(f"{BARRA_GRAFICO} — Test ({ZOOM_DIAS} días) — S+C-LN")
ax.set_ylabel("USD/MWh")
ax.legend()
plt.tight_layout()
plt.show()

**Interpretación:** el MASE está por debajo de 1 en 7 de las 8 barras (P.MONTT es la excepción,
MASE≈0.96), es decir, el modelo supera a la persistencia de una hora en casi todos los casos —
la señal honesta que pedía Jofré, sin forzarla.

## 3. P+C vs. S+C (con Diebold-Mariano)

Comparación entre la estrategia híbrida (Prophet + Clima, con *stacking*) y la directa
(Sol + Clima), para ambas arquitecturas (LN y DyT). El test de Diebold-Mariano usa varianza
HAC (Newey-West) para tolerar la autocorrelación de errores horarios — reemplaza a Wilcoxon,
que asume independencia.

In [ ]:
pc_vs_sc = cargar_json("pc_vs_sc_dm_test.json")

filas = []
for arq in ["LN", "DyT"]:
    for barra in BARRAS:
        d = pc_vs_sc[arq][barra]
        filas.append({
            "Arquitectura": arq, "Barra": barra,
            "MAE_P+C": d["mae_pc"], "MAE_S+C": d["mae_sc"], "Ganador": d["ganador_mae"],
            "DM_stat": d["DM"], "p_valor": d["p_valor"], "Significativo(5%)": d["sig"],
        })
df_pc_vs_sc = pd.DataFrame(filas).set_index(["Arquitectura", "Barra"]).round(4)
df_pc_vs_sc

In [ ]:
n_gana_sc = (df_pc_vs_sc["Ganador"] == "S+C").sum()
n_sig = df_pc_vs_sc["Significativo(5%)"].sum()
print(f"S+C gana en {n_gana_sc}/{len(df_pc_vs_sc)} combinaciones barra x arquitectura "
      f"({n_sig} de ellas estadísticamente significativas al 5%)")

**Interpretación:** S+C gana en 14 de 16 combinaciones; TARAPACA-LN es la única barra donde
P+C gana con significancia (el resto de las excepciones no son significativas), consistente
con el hallazgo central de la tesis (S+C > P+C) ya validado con selección hecha en validación,
no en test.

## 4. LN vs. DyT

Comparación directa de las dos arquitecturas de normalización (LayerNorm vs. Dynamic Tanh)
sobre S+C, mismo conjunto de test. El test DM se calcula en vivo aquí mismo (es una operación
barata sobre arrays ya cargados, no reentrena nada).

In [ ]:
filas = []
for barra in BARRAS:
    _, y_real_ln, y_pred_ln = cargar_eval_pkl(EXP_SOL_CLIMA["LN"], "LN", barra, "test")
    _, y_real_dyt, y_pred_dyt = cargar_eval_pkl(EXP_SOL_CLIMA["DyT"], "DyT", barra, "test")

    err_ln = y_real_ln - y_pred_ln
    err_dyt = y_real_dyt - y_pred_dyt
    n = min(len(err_ln), len(err_dyt))

    dm = diebold_mariano_desde_errores(np.abs(err_ln[:n]), np.abs(err_dyt[:n]))
    mae_ln, mae_dyt = np.abs(err_ln).mean(), np.abs(err_dyt).mean()
    filas.append({
        "Barra": barra, "MAE_LN": mae_ln, "MAE_DyT": mae_dyt,
        "Ganador": "LN" if mae_ln < mae_dyt else "DyT",
        "DM_stat": dm["DM"], "p_valor": dm["p_value"], "Significativo(5%)": dm["p_value"] < 0.05,
    })
df_ln_vs_dyt = pd.DataFrame(filas).set_index("Barra").round(4)
df_ln_vs_dyt

**Interpretación:** LN y DyT quedan prácticamente empatados en magnitud (diferencias de MAE
de centésimas a unas pocas décimas de USD/MWh); donde el DM test marca significancia, LN gana
más veces que DyT, pero ninguna arquitectura domina de forma sistemática — tal como anticipaba
Jofré.

## 5. Conformal Prediction

Los 5 métodos evaluados (SCP, LW-SCP, ACI, LW-ACP, CQR), ya con el piso $\hat\sigma_h$
(`eps=1.0`), la ventana de precio de falla excluida y los límites inferiores truncados en 0
(el precio no puede ser negativo).

In [ ]:
cp = cargar_json("cp_metodos_8barras.json")
metodos = ["scp", "lwscp", "aci", "lwacp", "cqr"]

filas = []
for barra in BARRAS:
    for metodo in metodos:
        d = cp[barra][metodo]
        filas.append({"Barra": barra, "Metodo": metodo.upper(),
                       "Cobertura": d["coverage"], "Ancho": d["width"]})
df_cp = pd.DataFrame(filas).set_index(["Barra", "Metodo"]).round(4)
df_cp

In [ ]:
winkler = cargar_json("cp_winkler_cobertura_condicional.json")
df_winkler = pd.DataFrame(winkler["winkler"], index=BARRAS).round(2)
print("Winkler score por método y barra:")
display(df_winkler)

print("\nPromedio Winkler por método (menor = mejor):")
display(df_winkler.mean().sort_values())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

cov_hora_lwacp = pd.Series({int(k): v for k, v in winkler["cov_hora_lwacp"].items()}).sort_index()
axes[0].plot(cov_hora_lwacp.index, cov_hora_lwacp.values, marker="o")
axes[0].axhline(0.95, color="gray", linestyle="--", linewidth=1)
axes[0].set_title("LW-ACP: cobertura por hora del día")
axes[0].set_xlabel("Hora")
axes[0].set_ylabel("Cobertura")

if "cov_mes_lwacp" in winkler:
    cov_mes_lwacp = pd.Series(winkler["cov_mes_lwacp"])
    axes[1].plot(range(len(cov_mes_lwacp)), cov_mes_lwacp.values, marker="o", color="darkorange")
    axes[1].axhline(0.95, color="gray", linestyle="--", linewidth=1)
    axes[1].set_title("LW-ACP: cobertura por mes")
    axes[1].set_xlabel("Mes (índice)")

plt.tight_layout()
plt.show()

**Interpretación:** LW-ACP tiene el menor Winkler score promedio (mejor balance ancho/cobertura)
entre los 5 métodos, y es más estable mes a mes que hora a hora — su $\alpha_t$ online sigue la
deriva estacional, pero al ser un escalar global no corrige subcobertura específica de ciertas
horas (7-8h y 19-21h, las transiciones de luz solar).

## 6. Puerto Montt + comparación con el CEN

Puerto Montt es el caso más difícil de la tesis (mayor volatilidad hidroeléctrica). Se compara
además contra el costo marginal programado que publica el propio Coordinador Eléctrico Nacional
(CEN) — la referencia institucional de facto — para 6 de las 8 barras (ATACAMA y TARAPACA no
tienen barra de costo marginal programado en la API pública del CEN).

In [ ]:
cen = pd.read_csv(os.path.join(RESULTADOS, "cen_comparacion.csv")).set_index("Barra").round(3)
cen

In [ ]:
fechas_pm, y_real_pm, y_pred_pm = cargar_eval_pkl(EXP_SOL_CLIMA["LN"], "LN", "P.MONTT", "test")
mae_pm = np.abs(y_real_pm - y_pred_pm).mean()
print(f"P.MONTT — MAE test (S+C-LN): {mae_pm:.2f} USD/MWh")
print(f"Mejora promedio del TFT sobre el CMg programado del CEN (6 barras): {cen['Mejora_%'].mean():.1f}%")

**Interpretación:** la mejora promedio de 55.5% frente al costo marginal *programado* del CEN
(que el propio Coordinador usa como referencia operativa) es una validación institucional
externa, no solo una comparación contra otros modelos de la literatura. Pendiente: completar
ATACAMA/TARAPACA si el usuario encuentra su mnemónico navegando el sitio del CEN.

## 7. Líneas base ingenuas + evaluación económica (pérdida asimétrica)

Persistencia, naive estacional (24h/168h) y AR vía OLS, todas sobre la serie cruda de test.
Más una función de pérdida linlin asimétrica ($\tau \neq 0.5$ penaliza más subestimar o
sobreestimar el precio, relevante para generadores/compradores) para evaluar si la ventaja del
TFT sobre la persistencia es robusta a esa asimetría.

In [ ]:
baselines = cargar_json("baselines_naive.json")
filas = []
for barra in BARRAS:
    d = baselines[barra]
    filas.append({
        "Barra": barra,
        "Persistencia_1h": d["persistencia"]["MAE"],
        "Naive_24h": d["naive_lag24"]["MAE"],
        "Naive_168h": d["naive_lag168"]["MAE"],
        "AR24": d["AR24"]["MAE"],
        "AR168": d["AR168"]["MAE"],
    })
df_baselines = pd.DataFrame(filas).set_index("Barra").round(3)
df_baselines

In [ ]:
perdida = pd.read_csv(os.path.join(RESULTADOS, "perdida_asimetrica.csv")).set_index("Barra")
cols_skill = [c for c in perdida.columns if c.startswith("skill_tau")]
df_skill = perdida[cols_skill].round(3)
df_skill.columns = [c.replace("skill_tau", "tau=") for c in df_skill.columns]
df_skill

In [ ]:
n_robusto = (df_skill > 0).all(axis=1).sum()
print(f"Barras con skill score positivo en las 5 tau evaluadas: {n_robusto}/8")

**Interpretación:** el TFT S+C-LN supera a la persistencia con pérdida asimétrica positiva
en 7 de 8 barras para todo el rango $\tau \in [0.2, 0.8]$ — la ventaja no depende de que el
error de subestimación y sobreestimación se penalicen igual. P.MONTT es la excepción esperada
(mismo patrón que en MASE): con $\tau$ bajo (penaliza sobreestimar) el skill score es
levemente negativo, y se vuelve positivo recién con $\tau \geq 0.35$.

## 8. Granularidad + resumen de ablación

¿Ayudan los datos más finos (15 min) frente a horario/diario? Y ¿cuánto aportan las features
de clima + posición solar frente a un modelo que solo ve la serie de precios?

In [ ]:
gran = cargar_json("granularidad_metricas.json")["metricas"]
filas = []
for gran_id, nombre in [("D", "Día"), ("h", "Horario"), ("15min", "15 min")]:
    for arq in ["LN", "DyT"]:
        for barra in ["ATACAMA", "CARDONES", "P.MONTT"]:
            key = f"{gran_id}_{arq}_{barra}"
            if key in gran:
                filas.append({"Granularidad": nombre, "Arquitectura": arq, "Barra": barra,
                               "MAE": gran[key]["MAE"], "R2": gran[key]["R2"], "n": gran[key]["n"]})
df_gran = pd.DataFrame(filas).set_index(["Granularidad", "Arquitectura", "Barra"]).round(3)
df_gran

In [ ]:
ablacion = cargar_json("ablacion_sin_features.json")
filas = []
for barra in ["ATACAMA", "CRUCERO", "P.MONTT", "TARAPACA"]:
    mae_sin = ablacion[barra]["MAE"]
    mae_con = sc_ln[barra]["test"]["MAE"]
    filas.append({"Barra": barra, "MAE_sin_features": mae_sin, "MAE_con_features": mae_con,
                   "Mejora_%": 100 * (mae_sin - mae_con) / mae_sin})
df_ablacion = pd.DataFrame(filas).set_index("Barra").round(3)
df_ablacion

**Interpretación:** 15 minutos reduce el MAE frente a horario en ATACAMA y P.MONTT, pero no
en CARDONES (el confound de período distinto entre granularidades limita la comparación directa,
ver Metodología). Las features de clima + posición solar aportan una mejora de 2.8% a 12.9%
según la barra — mayor en las barras solares (ATACAMA, CRUCERO) que en la hidroeléctrica
(P.MONTT), como es esperable.